# Types Shared by the Notebooks of Chapters 4 and 5

In OCaml, every type definition creates a *new* type: if a notebook that defines the type `label` is loaded
twice, the values created with the first definition have a type that is different from the second one, although
both types look the same.  Therefore, the types used by more than one notebook are defined in this notebook,
which is loaded via `run_notebook "Types.ipynb"`.  Since `run_notebook` loads every notebook only once, these
types are only defined once.  The notebooks that use these types show their definitions with the directive
`#show_type`.

In [ ]:
#use "../lib/nb.ml";;

## Regular Expressions

Regular expressions are represented as *nested tuples*, i.e. as values of the type `nested` that is defined in
the file `../lib/nested.ml`:
```
type value =
  | Int   of int
  | Float of float
  | Str   of string
  | Bool  of bool

type nested =
  | Leaf of value
  | Node of nested list
```
The type `regexp` is just another name for `nested`:
- $\emptyset$ is represented by the number `0`, i.e. by `Leaf (Int 0)`,
- $\varepsilon$ is represented by the string `"𝜀"`, i.e. by `Leaf (Str "𝜀")`,
- a character `c` is represented by the string `"c"`, i.e. by `Leaf (Str "c")`,
- $r^*$ is represented by the pair `(r, "*")`, i.e. by `Node [r; Leaf (Str "*")]`,
- $r_1 \cdot r_2$ is represented by the triple `(r1, "⋅", r2)`, and
- $r_1 + r_2$ is represented by the triple `(r1, "+", r2)`.

The notebooks print regular expressions in tuple notation, e.g. $(\texttt{a} + \texttt{b})^*$ is printed as
`(("a", "+", "b"), "*")`.  The functions `empty`, `epsilon`, `char`, `star`, `product`, and `sum` build regular
expressions.  To take a regular expression apart, we use *pattern matching*: the pattern
`Node [r1; Leaf (Str "+"); r2]` matches every sum and binds `r1` and `r2` to its summands.

In [ ]:
type regexp = nested

let empty   : regexp = Leaf (Int 0)
let epsilon : regexp = Leaf (Str "𝜀")

let char (c : char) : regexp = Leaf (Str (String.make 1 c))

let star    (r : regexp)               : regexp = Node [ r; Leaf (Str "*") ]
let product (r1 : regexp) (r2 : regexp) : regexp = Node [ r1; Leaf (Str "⋅"); r2 ]
let sum     (r1 : regexp) (r2 : regexp) : regexp = Node [ r1; Leaf (Str "+"); r2 ]

## Finite State Machines

The states of a finite state machine can be values of any type: the test notebooks use strings and
numbers, the function `nfa2dfa` produces machines whose states are *sets* of states, the function `minimize`
produces machines whose states are sets of states again, and the product automata of the notebook
`09-Equivalence.ipynb` have *pairs* of states.  Therefore, the types below are *polymorphic*: they have a
type parameter `'q`, which is the type of the states.  For example, `int nfa` is the type of the
non-deterministic finite state machines whose states are integers.

The transitions of a non-deterministic finite state machine are labelled either with a character or with
$\varepsilon$.  The labels have the type `label`: a label is either `Eps` or `Sym c`, where `c` is a character.

A finite state machine is a quintuple $\langle Q, \Sigma, \delta, q_0, A \rangle$.  We represent it as a
*record*, whose components are accessed by their names, e.g. `f.states`:
- `states` is the set $Q$ of states,
- `sigma` is the alphabet $\Sigma$,
- `delta` is the transition function $\delta$,
- `start` is the start state $q_0$, and
- `accepting` is the set $A$ of accepting states.

Sets are represented by the type `'a FSet.t` and dictionaries by the type `('k, 'v) Dict.t`, which are defined in
the file `../lib/fset.ml`.  The transition function of an *Nfa* maps a pair consisting of a state and a label
to a *set* of states, while the transition function of a *Dfa* maps a pair consisting of a state and a character to a
*single* state.

**Type safety:** Since *Nfa*s and *Dfa*s have different types, the compiler rejects an *Nfa* where a *Dfa* is
expected.

In [ ]:
type label =
  | Eps
  | Sym of char

type 'q trans_rel     = ('q * label, 'q FSet.t) Dict.t    (* transition relation of an nfa *)
type 'q trans_rel_det = ('q * char, 'q) Dict.t            (* transition function of a dfa  *)

type 'q nfa = {
  states    : 'q FSet.t;
  sigma     : char FSet.t;
  delta     : 'q trans_rel;
  start     : 'q;
  accepting : 'q FSet.t;
}

type 'q dfa = {
  states    : 'q FSet.t;
  sigma     : char FSet.t;
  delta     : 'q trans_rel_det;
  start     : 'q;
  accepting : 'q FSet.t;
}

Since the types `nfa` and `dfa` have the same field names, the compiler uses the type of the context to find
out which of the two types is meant.  Therefore, we annotate the variables and functions whose values are
finite state machines with their types, e.g. `let f : int nfa = { ... }`.

Finally, the function `not_yet_defined` is used to initialize *forward references*, which are explained in the
notebook `RegExp-Parser.ipynb`.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"